# Codomax Data Science Internship — Module 5

## Data Science Mini Project

**Duration:** Day 17 – Day 20  
**Level:** High  
**Priority:** Medium

### Project Title
# Student Performance Analysis — UCI Student Performance Dataset

### Objective
Analyze a real-world student dataset, clean it, visualize patterns, and summarize meaningful insights.

### Dataset Source
UCI Machine Learning Repository — Student Performance Dataset.

The dataset contains student grades along with demographic, social, and school-related variables.

> Run all cells from top to bottom in Google Colab.


# Day 17 — Setup and Load Dataset

For a reliable public source, this notebook uses the UCI Student Performance dataset.

We will analyze the **Math course dataset** (`student-mat.csv`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

## Load the Dataset

The UCI dataset is commonly distributed as a ZIP file containing:
- `student-mat.csv`
- `student-por.csv`

The code below downloads and loads the Mathematics dataset.


In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
zip_path = "/content/student_performance.zip"
extract_path = "/content/student_performance"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

for root, dirs, files in os.walk(extract_path):
    for f in files:
        if f == "student-mat.csv":
            csv_path = os.path.join(root, f)

df = pd.read_csv(csv_path, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

# Day 17 — Understand the Dataset


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset info:")
df.info()

In [ ]:
print("Summary statistics:")
df.describe().T

## Important Variables Used in This Project

- `G1` — First period grade
- `G2` — Second period grade
- `G3` — Final grade
- `studytime` — Weekly study time category
- `failures` — Number of past class failures
- `absences` — Number of school absences
- `internet` — Internet access at home
- `higher` — Wants to take higher education
- `schoolsup` — Extra educational support
- `famsup` — Family educational support


# Day 18 — Data Quality and Cleaning

Check missing values and duplicates before analysis.


In [ ]:
print("Missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nTotal missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

### Remove Exact Duplicates

We remove exact duplicate rows if they exist.


In [ ]:
df_clean = df.drop_duplicates().copy()

print("Rows before cleaning:", len(df))
print("Rows after duplicate removal:", len(df_clean))

## Create Useful Derived Columns


In [ ]:
df_clean["Pass_Status"] = np.where(df_clean["G3"] >= 10, "Pass", "Fail")

df_clean["Grade_Band"] = pd.cut(
    df_clean["G3"],
    bins=[-1, 9, 11, 13, 15, 20],
    labels=["Fail", "10-11", "12-13", "14-15", "16-20"]
)

df_clean[["G3", "Pass_Status", "Grade_Band"]].head()

# Day 18 — Basic Analysis


In [ ]:
print("Average Final Grade:", round(df_clean["G3"].mean(), 2))
print("Median Final Grade:", df_clean["G3"].median())
print("Highest Final Grade:", df_clean["G3"].max())
print("Lowest Final Grade:", df_clean["G3"].min())

pass_rate = (df_clean["Pass_Status"] == "Pass").mean() * 100
print("Pass Rate:", round(pass_rate, 2), "%")

# Day 19 — Data Visualization


## 1. Final Grade Distribution


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df_clean["G3"], bins=10, kde=True)
plt.title("Distribution of Final Grades")
plt.xlabel("Final Grade (G3)")
plt.ylabel("Number of Students")
plt.show()

## 2. Average Final Grade by Study Time


In [ ]:
study_analysis = (
    df_clean.groupby("studytime", as_index=False)["G3"]
    .mean()
)

plt.figure(figsize=(8,5))
sns.barplot(data=study_analysis, x="studytime", y="G3")
plt.title("Average Final Grade by Weekly Study Time Category")
plt.xlabel("Study Time Category")
plt.ylabel("Average Final Grade")
plt.show()

study_analysis

## 3. Absences vs Final Grade


In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df_clean, x="absences", y="G3")
plt.title("School Absences vs Final Grade")
plt.xlabel("Absences")
plt.ylabel("Final Grade")
plt.show()

print("Correlation:",
      round(df_clean["absences"].corr(df_clean["G3"]), 3))

## 4. Previous Failures vs Final Grade


In [ ]:
failure_analysis = (
    df_clean.groupby("failures", as_index=False)["G3"]
    .mean()
)

plt.figure(figsize=(8,5))
sns.barplot(data=failure_analysis, x="failures", y="G3")
plt.title("Average Final Grade by Number of Previous Failures")
plt.xlabel("Previous Failures")
plt.ylabel("Average Final Grade")
plt.show()

failure_analysis

## 5. Internet Access and Final Grade


In [ ]:
internet_analysis = (
    df_clean.groupby("internet", as_index=False)["G3"]
    .agg(["mean", "count"])
    .reset_index()
)

plt.figure(figsize=(7,5))
sns.barplot(data=df_clean, x="internet", y="G3")
plt.title("Average Final Grade by Home Internet Access")
plt.xlabel("Internet Access at Home")
plt.ylabel("Average Final Grade")
plt.show()

internet_analysis

## 6. Higher Education Aspiration and Final Grade


In [ ]:
higher_analysis = (
    df_clean.groupby("higher", as_index=False)["G3"]
    .mean()
)

plt.figure(figsize=(7,5))
sns.barplot(data=higher_analysis, x="higher", y="G3")
plt.title("Average Final Grade by Higher Education Aspiration")
plt.xlabel("Wants Higher Education")
plt.ylabel("Average Final Grade")
plt.show()

higher_analysis

## 7. Correlation Heatmap


In [ ]:
selected_numeric = df_clean[
    ["age", "studytime", "failures", "absences", "G1", "G2", "G3"]
]

plt.figure(figsize=(8,6))
sns.heatmap(selected_numeric.corr(), annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# Day 20 — Insight Generation

The following cells compute evidence-based findings from the dataset.


In [ ]:
top_corr = (
    selected_numeric.corr()["G3"]
    .drop("G3")
    .sort_values(key=abs, ascending=False)
)

print("Variables most correlated with final grade:")
print(top_corr)

In [ ]:
print("KEY FINDINGS")
print("-" * 60)

print("1. Total student records analyzed:", len(df_clean))
print("2. Average final grade:", round(df_clean["G3"].mean(), 2))
print("3. Pass rate:", round((df_clean["Pass_Status"] == "Pass").mean()*100, 2), "%")
print("4. Correlation of G1 with G3:", round(df_clean["G1"].corr(df_clean["G3"]), 3))
print("5. Correlation of G2 with G3:", round(df_clean["G2"].corr(df_clean["G3"]), 3))
print("6. Correlation of absences with G3:", round(df_clean["absences"].corr(df_clean["G3"]), 3))
print("7. Correlation of failures with G3:", round(df_clean["failures"].corr(df_clean["G3"]), 3))

# Interpretation Guide

Use the generated results to write your own observations.

Possible discussion areas:

- Whether earlier-period grades are strongly related to final grades
- Whether previous failures are associated with lower final grades
- Whether absence levels show a meaningful relationship with final performance
- Whether study-time categories differ in average final grade
- Whether students aspiring to higher education show different average outcomes
- Whether internet access is associated with different average grades

Avoid claiming **causation** from simple correlations or group comparisons.


# Final Project Summary

## Project
**Student Performance Analysis using Python**

## Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn

## Workflow
**Load → Inspect → Clean → Transform → Analyze → Visualize → Interpret**

## Main Tasks Completed
- Loaded a real public dataset
- Checked data quality
- Removed duplicates
- Created derived variables
- Calculated descriptive statistics
- Compared student groups
- Measured correlations
- Built multiple visualizations
- Generated data-backed findings


# Export Cleaned Dataset


In [ ]:
df_clean.to_csv("student_performance_cleaned.csv", index=False)
print("Saved: student_performance_cleaned.csv")

# Practice / Improvement Tasks

Before final submission, try at least three:

1. Compare male and female students' average final grade.
2. Analyze parental education (`Medu`, `Fedu`) vs final grade.
3. Compare students receiving school support vs those who do not.
4. Compare family support vs final grade.
5. Analyze free-time or going-out variables.
6. Create one additional visualization.
7. Write five concise findings in your own words.


# GitHub Submission Structure

Recommended repository structure:

```text
Codomax-Data-Science-Internship/
│
├── Module-1/
├── Module-2/
├── Module-3/
├── Module-4/
└── Module-5/
    ├── Codomax_Module_5_Data_Science_Mini_Project.ipynb
    └── student_performance_cleaned.csv
```

You may also add a `README.md` describing:
- Project objective
- Dataset source
- Tools used
- Analysis steps
- Key findings


# Module 5 Submission Checklist

- [ ] Run every notebook cell successfully
- [ ] Review all visualizations
- [ ] Add your own written insights
- [ ] Save/export the cleaned dataset
- [ ] Upload the notebook to your GitHub repository
- [ ] Upload the cleaned CSV if desired
- [ ] Add/update README.md
- [ ] Copy your GitHub repository link
- [ ] Publish your Module 5 LinkedIn post
- [ ] Copy your LinkedIn post link

## Submission Deliverables

**GitHub Repository Link:**  
`Paste your GitHub repository link here`

**LinkedIn Post Link:**  
`Paste your LinkedIn post link here`
